# SmartAid AI — XGBoost Severity Classification

This notebook trains and evaluates an XGBoost multiclass classifier using the feature design documented for SmartAid AI: symptom text, age, symptom duration, sex, emergency category, and 15 binary symptom indicators.

**Important:** The project audit describes a 10,000-row synthetic dataset. Results on synthetic data do not establish clinical validity. This notebook is for academic development and evaluation, not medical diagnosis.

## Use in Google Colab
1. Open the notebook in Colab.
2. Run the dependency installation cell.
3. Upload `smartaid_ai_test_dataset1.csv`.
4. Check the printed columns and adjust the target/feature configuration if needed.
5. Run the remaining cells in order.

## 1. Install dependencies

In [ ]:
%pip -q install xgboost scikit-learn pandas numpy matplotlib joblib

## 2. Upload the CSV dataset

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Please upload a CSV file.")
CSV_PATH = csv_files[0]
print("Using:", CSV_PATH)

## 3. Load and inspect the dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

## 4. Configure target and feature columns

The target column is auto-detected from common names. Confirm that it is the severity label. If not, set `TARGET_COLUMN` to the exact name shown above. Update feature names if your CSV uses different spellings.

In [ ]:
TARGET_CANDIDATES = [
    "severity_level", "severity", "severity_label", "target", "label",
    "triage_level", "risk_level"
]
TARGET_COLUMN = next((c for c in TARGET_CANDIDATES if c in df.columns), None)
if TARGET_COLUMN is None:
    raise ValueError("Set TARGET_COLUMN to the exact severity-label column in your CSV.")

TEXT_COLUMNS = ["symptom_text"]
NUMERIC_COLUMNS = ["age", "symptom_duration_min"]
CATEGORICAL_COLUMNS = ["sex", "emergency_category"]
BINARY_COLUMNS = [
    "chest_pain", "difficulty_breathing", "sweating", "nausea", "dizziness",
    "unconsciousness", "bleeding", "burn", "fever", "cough", "seizure",
    "abdominal_pain", "headache", "weakness", "injury"
]
FEATURE_COLUMNS = TEXT_COLUMNS + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS + BINARY_COLUMNS
missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Configured feature columns missing from CSV: {missing}. Edit the lists above.")
print("Target:", TARGET_COLUMN)
print("Feature columns:", len(FEATURE_COLUMNS))
print("Classes:", sorted(df[TARGET_COLUMN].dropna().astype(str).unique()))

## 5. Prepare features and target

In [ ]:
df = df.dropna(subset=[TARGET_COLUMN]).copy()
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].astype(str).str.strip()

# TF-IDF requires strings; convert missing symptom narratives to empty text.
X[TEXT_COLUMNS[0]] = X[TEXT_COLUMNS[0]].fillna("").astype(str)

for col in BINARY_COLUMNS:
    X[col] = pd.to_numeric(X[col], errors="coerce")

print("Feature matrix:", X.shape)
display(y.value_counts().rename_axis("severity_class").to_frame("count"))

## 6. Stratified 80/20 train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

## 7. Build the preprocessing and XGBoost pipeline

Preprocessing is inside the scikit-learn pipeline to reduce data leakage. `multi:softprob` is used to make class scores available for inspection. The estimator settings follow the project audit's documented values where applicable.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            ngram_range=(1, 2), max_features=300, stop_words="english"
        ), TEXT_COLUMNS[0]),
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), NUMERIC_COLUMNS),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), CATEGORICAL_COLUMNS),
        ("binary", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent"))
        ]), BINARY_COLUMNS)
    ],
    remainder="drop"
)

classifier = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    min_child_weight=4,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", classifier)
])
print("Pipeline ready.")

## 8. Train the model with balanced sample weights

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
pipeline.fit(X_train, y_train, classifier__sample_weight=sample_weights)
print("Training complete.")

## 9. Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score
)
import matplotlib.pyplot as plt

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Test accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(
    ax=ax, values_format="d", colorbar=False
)
plt.title("SmartAid AI — XGBoost Confusion Matrix")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 10. Five-fold stratified cross-validation on training data

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    pipeline, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=1
)
print("Fold macro-F1 scores:", np.round(cv_scores, 4))
print(f"Mean macro-F1: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")

## 11. Inspect one prediction and its class scores

In [ ]:
example = X_test.iloc[[0]]
predicted_class = pipeline.predict(example)[0]
probabilities = pipeline.predict_proba(example)[0]
class_names = pipeline.named_steps["classifier"].classes_

print("Predicted class:", predicted_class)
print("Highest class score:", f"{probabilities.max()*100:.2f}%")
display(
    pd.DataFrame({"class": class_names, "model_score": probabilities})
      .sort_values("model_score", ascending=False)
      .reset_index(drop=True)
)
display(example)

## 12. Save the pipeline and metrics

In [ ]:
import joblib, json, os
from datetime import datetime, timezone

MODEL_PATH = "severity_model_pipeline.joblib"
METRICS_PATH = "severity_model_metrics.json"
joblib.dump(pipeline, MODEL_PATH)

metrics = {
    "model": "XGBoostClassifier",
    "dataset_file": os.path.basename(CSV_PATH),
    "target_column": TARGET_COLUMN,
    "feature_columns": FEATURE_COLUMNS,
    "target_classes": [str(c) for c in pipeline.named_steps["classifier"].classes_],
    "test_rows": int(len(y_test)),
    "test_accuracy": float(accuracy),
    "test_f1_macro": float(macro_f1),
    "test_f1_weighted": float(weighted_f1),
    "cv_f1_macro_scores": [float(s) for s in cv_scores],
    "cv_f1_macro_mean": float(cv_scores.mean()),
    "cv_f1_macro_std": float(cv_scores.std()),
    "hyperparameters": {
        "n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
        "min_child_weight": 4, "objective": "multi:softprob",
        "eval_metric": "mlogloss"
    },
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_note": "Synthetic dataset; metrics do not establish clinical validity."
}
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", MODEL_PATH)
print("Saved:", METRICS_PATH)

## 13. Download model artifacts

In [ ]:
from google.colab import files
files.download("severity_model_pipeline.joblib")
files.download("severity_model_metrics.json")

## 14. Integration notes

- The FastAPI backend must use the same feature names, preprocessing expectations, and class labels as this saved pipeline.
- This notebook trains directly on string labels, so the classifier's `classes_` contains the class names. Check whether your current backend actually needs a separate `label_encoder.joblib` before replacing existing artifacts.
- Compare the new metrics with your current report before replacing anything in your repository.
- This notebook does not deploy the model or modify your local repository.
- Because the dataset is synthetic, this is an academic prototype and not a clinically validated triage system.